# FruitScan AI — Research Notebook

This notebook documents the Fruits-360 dataset exploration, model training and evaluation.
Before running, download the dataset from https://www.kaggle.com/datasets/moltean/fruits and extract it to `data/fruits-360/`.

The model is trained by `src/train.py`. This notebook inspects the data, runs the reproducible training pipeline and displays the real metrics produced by that run. Do not report placeholder or invented metrics.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
from PIL import Image
from collections import Counter

DATA_DIR = Path("../data/fruits-360")
TRAIN_DIR = DATA_DIR / "Training"
TEST_DIR = DATA_DIR / "Test"

assert TRAIN_DIR.exists() and TEST_DIR.exists(), (
    "Dataset folders not found. Extract Fruits-360 into ../data/fruits-360/"
)

train_classes = sorted(p.name for p in TRAIN_DIR.iterdir() if p.is_dir())
test_classes = sorted(p.name for p in TEST_DIR.iterdir() if p.is_dir())
common_classes = sorted(set(train_classes) & set(test_classes))

print("Training classes:", len(train_classes))
print("Test classes:", len(test_classes))
print("Common classes:", len(common_classes))
print("Example classes:", common_classes[:20])

In [ ]:
# Select a manageable subset of classes for the course project.
SELECTED_CLASSES = [
    "Apple Braeburn", "Apple Golden 1", "Apple Granny Smith",
    "Banana", "Cherry 1", "Grape Blue", "Kiwi", "Lemon",
    "Mango", "Orange", "Peach", "Pear", "Pineapple",
    "Strawberry", "Watermelon"
]
SELECTED_CLASSES = [c for c in SELECTED_CLASSES if c in common_classes]
print(f"Selected {len(SELECTED_CLASSES)} classes:")
print(SELECTED_CLASSES)
assert len(SELECTED_CLASSES) >= 2, "Check selected folder names against the dataset." 

In [ ]:
# Count images in each selected class.
train_counts = {
    c: len(list((TRAIN_DIR / c).glob("*")))
    for c in SELECTED_CLASSES
}
test_counts = {
    c: len(list((TEST_DIR / c).glob("*")))
    for c in SELECTED_CLASSES
}

plt.figure(figsize=(14, 5))
plt.bar(train_counts.keys(), train_counts.values())
plt.xticks(rotation=80, ha="right")
plt.title("Training images per selected class")
plt.ylabel("Number of images")
plt.tight_layout()
plt.show()

plt.figure(figsize=(14, 5))
plt.bar(test_counts.keys(), test_counts.values())
plt.xticks(rotation=80, ha="right")
plt.title("Test images per selected class")
plt.ylabel("Number of images")
plt.tight_layout()
plt.show()

In [ ]:
# Display sample images from selected classes.
fig, axes = plt.subplots(3, 5, figsize=(14, 9))
for ax, class_name in zip(axes.flat, SELECTED_CLASSES[:15]):
    samples = list((TRAIN_DIR / class_name).glob("*"))
    if samples:
        image = Image.open(samples[0]).convert("RGB")
        ax.imshow(image)
        ax.set_title(class_name, fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Preprocessing and model configuration

The training pipeline resizes images to 100×100 RGB. Pixel normalization is included as a `Rescaling(1/255)` layer inside the CNN, so the inference pipeline must pass pixel values in the 0–255 range.

The script creates a validation split from the training directory and keeps the Kaggle `Test` directory for final evaluation. The selected classes must exist in both directories.

In [ ]:
# Run the reproducible training script from the repository root.
# If TensorFlow is not installed in this notebook environment, install requirements first.
import subprocess
from pathlib import Path

repo_root = Path("..").resolve()
command = [
    "python", "-m", "src.train",
    "--data-dir", str(repo_root / "data" / "fruits-360"),
    "--output-dir", str(repo_root / "artifacts"),
    "--epochs", "15",
    "--classes", *SELECTED_CLASSES
]
print("Running:", " ".join(command))
subprocess.run(command, cwd=repo_root, check=True)

In [ ]:
# Load actual evaluation results from the training run.
metrics_path = repo_root / "artifacts" / "metrics.json"
metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
print(f"Test accuracy: {metrics['test_accuracy']:.4f}")
print(f"Test loss: {metrics['test_loss']:.4f}")

In [ ]:
# Plot training and validation curves.
history_path = repo_root / "artifacts" / "training_history.json"
history = json.loads(history_path.read_text(encoding="utf-8"))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["accuracy"], label="Training")
axes[0].plot(history["val_accuracy"], label="Validation")
axes[0].set_title("Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history["loss"], label="Training")
axes[1].plot(history["val_loss"], label="Validation")
axes[1].set_title("Loss")
axes[1].set_xlabel("Epoch")
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# Display the confusion matrix generated by the training script.
cm_path = repo_root / "artifacts" / "confusion_matrix.png"
display(Image.open(cm_path))

## Experiment comparison and conclusions

For the second experiment, change one or more model hyperparameters in `src/train.py` (for example, number of filters, dropout, learning rate, or number of epochs), run training again into a different output directory, and compare the real metrics.

Record:
- architecture and hyperparameters;
- training time and hardware;
- test accuracy, precision, recall and F1;
- classes that are commonly confused;
- signs of overfitting or underfitting.

**Limitations:** Fruits-360 contains controlled images with isolated produce. Performance on phone photos with cluttered backgrounds, multiple objects, unusual lighting or classes outside the training list may be lower. Softmax scores are not calibrated probabilities and must not be presented as guaranteed correctness.